# RQ1 — General-Purpose VLMs on Astronaut Health Panels (Kaggle)

**Question:** How effectively do general-purpose Vision-Language Models perform on astronaut health monitoring tasks using publicly available spaceflight biomedical datasets?

| Model | Dataset |
|---|---|
| CLIP | NASA OSDR (Inspiration4) |
| BLIP-2 | NASA OSDR (Inspiration4) |
| LLaVA-1.6 | NASA GeneLab |
| Qwen2.5-VL | NASA GeneLab |

Metrics: Accuracy, Precision, Recall, F1, AUROC, MCC → `results/rq1_table1.csv`

### Caveats (read before citing numbers)
- OSDR Inspiration4 / GeneLab panels are **tabular**. This notebook **renders each row as a feature heatmap** so VLMs can score them.
- Labels are a **flight-phase proxy** (preflight `L-*` = 0, return `R+*` = 1), **not** a clinical endpoint.
- Results are exploratory plumbing for Table 1 — do not claim clinical detection.

### Kaggle / SSH
1. Enable **GPU** (T4 16GB or better).
2. Attach Inspiration4 so `panel.csv` is at `/kaggle/input/inspiration4/` **or** sync/clone this repo with `data/raw/inspiration4/panel.csv`.
3. Optionally attach GeneLab as `/kaggle/input/genelab_omics/panel.csv` for LLaVA / Qwen rows; otherwise those rows are `missing_data` (or set `FALLBACK_GENELAB_TO_INSPIRATION4=True`).
4. Run top → bottom. Download `results/rq1_table1.csv` when done.

In [ ]:
# 0. Config
REPO_URL = "https://github.com/theap06/shrimpy.git"
# If git clone fails (Internet off / GitHub blocked), Add Input a dataset with the
# repo contents, or set CODE_DATASET to that Kaggle input folder name.
CODE_DATASET = ""  # e.g. "shrimpy" if you uploaded the repo as a Kaggle dataset
FALLBACK_GENELAB_TO_INSPIRATION4 = True  # fill LLaVA/Qwen rows using Inspiration4 if no GeneLab
MAX_FEATURES = 64
SEED = 2026
# Explicit panel path if auto-detect fails (your current upload):
PANEL_CSV = "/kaggle/input/datasets/srinithyapaninga/algodata/panel.csv"

In [ ]:
# 1. Working directory (Kaggle — prefer uploaded code if git/DNS fails)
import os
import pathlib
import shutil
import subprocess
import sys

ON_KAGGLE = pathlib.Path("/kaggle/working").exists()

def _has_helpers(p: pathlib.Path) -> bool:
    return (p / "scripts" / "rq1_vlm_helpers.py").exists()

if ON_KAGGLE:
    candidates = [
        pathlib.Path("/kaggle/working/shrimpy"),
        pathlib.Path("/kaggle/working"),
    ]
    if CODE_DATASET:
        candidates.insert(0, pathlib.Path(f"/kaggle/input/{CODE_DATASET}"))
        candidates.insert(0, pathlib.Path(f"/kaggle/input/datasets/{CODE_DATASET}"))
    inp = pathlib.Path("/kaggle/input")
    if inp.exists():
        for hit in inp.rglob("rq1_vlm_helpers.py"):
            candidates.append(hit.parents[1])

    ROOT = next((c for c in candidates if _has_helpers(c)), None)

    if ROOT is None:
        ROOT = pathlib.Path("/kaggle/working/shrimpy")
        try:
            if not ROOT.exists():
                subprocess.run(
                    ["git", "clone", "--depth", "1", REPO_URL, str(ROOT)],
                    check=True,
                    timeout=120,
                )
        except Exception as exc:
            raise SystemExit(
                f"Could not get repo code (git failed: {exc!r}).\n"
                "1) Notebook settings → turn Internet ON (also required for model downloads).\n"
                "2) Or upload the GitHub repo as a Kaggle dataset and set CODE_DATASET.\n"
                "3) Or paste/upload scripts/rq1_vlm_helpers.py under /kaggle/working/scripts/."
            ) from exc

    if str(ROOT).startswith("/kaggle/input"):
        dest = pathlib.Path("/kaggle/working/shrimpy")
        if not _has_helpers(dest):
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(ROOT, dest)
        ROOT = dest

    os.chdir(ROOT)
else:
    ROOT = pathlib.Path.cwd()
    if not _has_helpers(ROOT):
        if _has_helpers(ROOT.parent):
            ROOT = ROOT.parent
            os.chdir(ROOT)
        else:
            raise SystemExit(f"Cannot find repo root from {pathlib.Path.cwd()}")

sys.path.insert(0, str(ROOT / "scripts"))
print("cwd:", pathlib.Path.cwd())
print("on_kaggle:", ON_KAGGLE)
print("helpers:", (ROOT / "scripts" / "rq1_vlm_helpers.py").exists())


In [ ]:
# 2. Dependencies (Kaggle already has torch + CUDA usually)
import subprocess
import sys

pkgs = [
    "transformers>=4.45.0",
    "accelerate>=0.33.0",
    "bitsandbytes>=0.43.0",
    "pillow",
    "matplotlib",
    "einops",
    "qwen-vl-utils",
    "sentencepiece",
    "protobuf",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Build heatmap image sets from panels
from pathlib import Path
import shutil

from rq1_vlm_helpers import (
    resolve_panel_path,
    render_panel_heatmaps,
    VALIDITY_WARNING,
)

OUT = Path("data/vlm_rq1")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
Path("results").mkdir(exist_ok=True)

# Prefer explicit PANEL_CSV (your algodata upload), then auto-detect
i4_panel = Path(PANEL_CSV) if PANEL_CSV and Path(PANEL_CSV).is_file() else None
if i4_panel is None:
    i4_panel = resolve_panel_path("inspiration4", Path.cwd())
    if i4_panel is None:
        i4_panel = resolve_panel_path("algodata", Path.cwd())

gl_panel = resolve_panel_path("genelab_omics", Path.cwd())
print("inspiration4/algodata panel:", i4_panel)
print("genelab_omics panel:", gl_panel)

if i4_panel is None:
    raise SystemExit(
        "panel.csv not found. Expected something like:\n"
        "  /kaggle/input/datasets/srinithyapaninga/algodata/panel.csv\n"
        "Set PANEL_CSV in the config cell to the path printed by:\n"
        "  !find /kaggle/input -name panel.csv"
    )

# Stage into data/raw so the rest of the pipeline is consistent
dst = Path("data/raw/inspiration4")
dst.mkdir(parents=True, exist_ok=True)
staged = dst / "panel.csv"
if Path(i4_panel).resolve() != staged.resolve():
    shutil.copy2(i4_panel, staged)
    print("copied", i4_panel, "->", staged)
i4_panel = staged

samples_i4 = render_panel_heatmaps(i4_panel, OUT, "inspiration4", max_features=MAX_FEATURES)
print(f"Inspiration4 images: {len(samples_i4)}  pos={sum(s['label'] for s in samples_i4)}")

samples_gl = []
if gl_panel is not None:
    samples_gl = render_panel_heatmaps(gl_panel, OUT, "genelab_omics", max_features=MAX_FEATURES)
    print(f"GeneLab images: {len(samples_gl)}  pos={sum(s['label'] for s in samples_gl)}")
elif FALLBACK_GENELAB_TO_INSPIRATION4:
    samples_gl = samples_i4
    print("WARN: GeneLab missing — falling back to Inspiration4 images for LLaVA/Qwen")
else:
    print("WARN: GeneLab missing — LLaVA/Qwen rows will be status=missing_data")

print(VALIDITY_WARNING)


In [ ]:
# 4. Shared inference + metrics utilities
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

from rq1_vlm_helpers import (
    POSITIVE_PROMPT,
    NEGATIVE_PROMPT,
    VQA_QUESTION,
    compute_table1_metrics,
    parse_yes_no,
    table1_row,
    empty_table1_row,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
table_rows = []
pred_path = Path("results/rq1_predictions.jsonl")
if pred_path.exists():
    pred_path.unlink()


def append_preds(model_name, samples, y_score, y_pred):
    with pred_path.open("a", encoding="utf-8") as f:
        for s, sc, pr in zip(samples, y_score, y_pred):
            f.write(json.dumps({
                "model": model_name,
                "id": s["id"],
                "dataset": s["dataset"],
                "label": s["label"],
                "score": float(sc),
                "pred": int(pr),
            }) + "\n")


def finalize_model(model_name, dataset_label, samples, y_score, y_pred=None):
    y_true = np.array([s["label"] for s in samples], dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if y_pred is None:
        y_pred = (y_score >= 0.5).astype(int)
    else:
        y_pred = np.asarray(y_pred, dtype=int)
    metrics = compute_table1_metrics(y_true, y_score, y_pred)
    row = table1_row(model_name, dataset_label, metrics, len(samples))
    table_rows.append(row)
    append_preds(model_name, samples, y_score, y_pred)
    print(pd.Series(row).to_string())
    return row


print("device:", DEVICE)
print("prompts:", NEGATIVE_PROMPT, "|", POSITIVE_PROMPT)

## CLIP — NASA OSDR (Inspiration4)

In [ ]:
# 5. CLIP zero-shot
from transformers import CLIPModel, CLIPProcessor

clip_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_name).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(clip_name)

text = clip_proc(
    text=[NEGATIVE_PROMPT, POSITIVE_PROMPT],
    return_tensors="pt",
    padding=True,
).to(DEVICE)

scores = []
with torch.inference_mode():
    for s in samples_i4:
        image = Image.open(s["image_path"]).convert("RGB")
        inputs = clip_proc(images=image, return_tensors="pt").to(DEVICE)
        out = clip_model(**inputs, **text)
        probs = out.logits_per_image.softmax(dim=-1).cpu().numpy()[0]
        scores.append(float(probs[1]))  # P(positive prompt)

finalize_model("CLIP", "NASA OSDR (Inspiration4)", samples_i4, scores)

del clip_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## BLIP-2 — NASA OSDR (Inspiration4)

In [ ]:
# 6. BLIP-2 VQA-style yes/no
from transformers import Blip2ForConditionalGeneration, Blip2Processor

blip_name = "Salesforce/blip2-opt-2.7b"
blip_proc = Blip2Processor.from_pretrained(blip_name)
blip_dtype = torch.float16 if DEVICE == "cuda" else torch.float32
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    blip_name,
    torch_dtype=blip_dtype,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    blip_model = blip_model.to(DEVICE)
blip_model.eval()

scores, preds = [], []
with torch.inference_mode():
    for s in samples_i4:
        image = Image.open(s["image_path"]).convert("RGB")
        inputs = blip_proc(images=image, text=VQA_QUESTION, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(dtype=blip_dtype)
        out_ids = blip_model.generate(**inputs, max_new_tokens=8)
        text = blip_proc.batch_decode(out_ids, skip_special_tokens=True)[0].strip()
        pred = parse_yes_no(text)
        if pred is None:
            pred, score = 0, 0.5
        else:
            score = 0.9 if pred == 1 else 0.1
        preds.append(pred)
        scores.append(score)

finalize_model("BLIP-2", "NASA OSDR (Inspiration4)", samples_i4, scores, preds)

del blip_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## LLaVA-1.6 — NASA GeneLab

In [ ]:
# 7. LLaVA-1.6 (4-bit on GPU)
llava_dataset_label = "NASA GeneLab"
llava_samples = samples_gl

if not llava_samples:
    table_rows.append(empty_table1_row("LLaVA-1.6", llava_dataset_label, "missing_data"))
    print("LLaVA-1.6 skipped — no GeneLab images. Set FALLBACK_GENELAB_TO_INSPIRATION4=True to override.")
else:
    from transformers import (
        LlavaNextProcessor,
        LlavaNextForConditionalGeneration,
        BitsAndBytesConfig,
    )

    llava_name = "llava-hf/llava-v1.6-mistral-7b-hf"
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
    ) if DEVICE == "cuda" else None

    llava_proc = LlavaNextProcessor.from_pretrained(llava_name)
    llava_model = LlavaNextForConditionalGeneration.from_pretrained(
        llava_name,
        quantization_config=bnb,
        device_map="auto" if DEVICE == "cuda" else None,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    if DEVICE != "cuda":
        llava_model = llava_model.to(DEVICE)
    llava_model.eval()

    scores, preds = [], []
    with torch.inference_mode():
        for s in llava_samples:
            image = Image.open(s["image_path"]).convert("RGB")
            prompt = f"[INST] <image>\n{VQA_QUESTION} [/INST]"
            inputs = llava_proc(prompt, image, return_tensors="pt").to(llava_model.device)
            out_ids = llava_model.generate(**inputs, max_new_tokens=16)
            gen = out_ids[:, inputs["input_ids"].shape[-1]:]
            text = llava_proc.batch_decode(gen, skip_special_tokens=True)[0].strip()
            pred = parse_yes_no(text)
            if pred is None:
                pred, score = 0, 0.5
            else:
                score = 0.9 if pred == 1 else 0.1
            preds.append(pred)
            scores.append(score)

    finalize_model("LLaVA-1.6", llava_dataset_label, llava_samples, scores, preds)
    del llava_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Qwen2.5-VL — NASA GeneLab

In [ ]:
# 8. Qwen2.5-VL-3B (fits T4 more reliably than 7B)
qwen_dataset_label = "NASA GeneLab"
qwen_samples = samples_gl

if not qwen_samples:
    table_rows.append(empty_table1_row("Qwen2.5-VL", qwen_dataset_label, "missing_data"))
    print("Qwen2.5-VL skipped — no GeneLab images.")
else:
    from transformers import (
        Qwen2_5_VLForConditionalGeneration,
        AutoProcessor,
        BitsAndBytesConfig,
    )

    qwen_name = "Qwen/Qwen2.5-VL-3B-Instruct"
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
    ) if DEVICE == "cuda" else None

    qwen_proc = AutoProcessor.from_pretrained(qwen_name)
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        qwen_name,
        quantization_config=bnb,
        device_map="auto" if DEVICE == "cuda" else None,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    if DEVICE != "cuda":
        qwen_model = qwen_model.to(DEVICE)
    qwen_model.eval()

    scores, preds = [], []
    with torch.inference_mode():
        for s in qwen_samples:
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": s["image_path"]},
                    {"type": "text", "text": VQA_QUESTION},
                ],
            }]
            chat = qwen_proc.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image = Image.open(s["image_path"]).convert("RGB")
            inputs = qwen_proc(
                text=[chat], images=[image], return_tensors="pt", padding=True
            )
            inputs = {
                k: v.to(qwen_model.device) if hasattr(v, "to") else v
                for k, v in inputs.items()
            }
            out_ids = qwen_model.generate(**inputs, max_new_tokens=16)
            trimmed = [o[len(i):] for i, o in zip(inputs["input_ids"], out_ids)]
            text = qwen_proc.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
            pred = parse_yes_no(text)
            if pred is None:
                pred, score = 0, 0.5
            else:
                score = 0.9 if pred == 1 else 0.1
            preds.append(pred)
            scores.append(score)

    finalize_model("Qwen2.5-VL", qwen_dataset_label, qwen_samples, scores, preds)
    del qwen_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Table 1 — compare models

In [ ]:
# 9. Write Table 1 + comparison
from pathlib import Path
import pandas as pd

cols = [
    "model", "dataset", "accuracy", "precision", "recall",
    "f1", "auroc", "mcc", "status", "n_samples", "validity_warning",
]
table = pd.DataFrame(table_rows)
for c in cols:
    if c not in table.columns:
        table[c] = float("nan")
table = table[cols]
out_csv = Path("results/rq1_table1.csv")
table.to_csv(out_csv, index=False)
print("Wrote", out_csv.resolve())
display(table)

done = table[table["status"] == "completed"].copy()
if len(done):
    print("\nRanked by F1:")
    print(
        done.sort_values("f1", ascending=False)[
            ["model", "dataset", "f1", "auroc", "mcc"]
        ].to_string(index=False)
    )
else:
    print("No completed model rows — check GPU / data setup.")